<p>
<img src="../imgs/EII-ULPGC-logo.jpeg" width="430px" align="right">

# **NOTEBOOK 2**
---

# **Modelos estadísticos del lenguaje**

## **Modelos markovianos**

Los modelos markovianos describen procesos en los que el futuro depende del estado actual y no de toda la historia anterior. Esta propiedad se conoce como propiedad de Markov. El modelo puede conservar una cantidad limitada de información del pasado, determinada por su orden.

### **Proceso de Markov**

Por ejemplo, el tiempo atmosférico de un día puede modelarse a partir del tiempo atmosférico del día anterior:

<DIV ALIGN="center">
<img src="imgs/markov.svg" width="50%">
</DIV>

La tabla siguiente representa las probabilidades de transición. La suma de cada fila es 1.

|          | soleado | nublado | lluvioso |
|----------|---------|---------|----------|
| **soleado**  | 0.6     | 0.3     | 0.1      |
| **nublado**  | 0.1     | 0.5     | 0.4      |
| **lluvioso** | 0.6     | 0.2     | 0.2      |

Para un proceso de Markov de primer orden:

$$ P(s_{t+1} \mid s_t) = P(s_{t+1} \mid s_t, s_{t-1}, \ldots, s_1) $$

Esto significa que, conocido el estado actual $s_t$, los estados anteriores no aportan información adicional para predecir $s_{t+1}$.

### **Cadena de Markov**

Una cadena de Markov es una secuencia de estados. Por ejemplo:

`soleado -> nublado -> nublado -> lluvioso`

La probabilidad de esta cadena es:

$$ P(soleado, nublado, nublado, lluvioso) = P(soleado) \cdot P(nublado \mid soleado) \cdot P(nublado \mid nublado) \cdot P(lluvioso \mid nublado) $$

Como la tabla indica que $P(nublado \mid soleado)=0.3$, $P(nublado \mid nublado)=0.5$ y $P(lluvioso \mid nublado)=0.4$, la probabilidad condicional de las transiciones es:

$$ P(nublado \mid soleado) \cdot P(nublado \mid nublado) \cdot P(lluvioso \mid nublado) = 0.3 \cdot 0.5 \cdot 0.4 = 0.06 $$

Si además suponemos que $P(soleado)=0.43$, la probabilidad conjunta de la cadena es:

$$ 0.43 \cdot 0.3 \cdot 0.5 \cdot 0.4 = 0.0258 $$

## **Modelos de lenguaje**

Un modelo de lenguaje asigna probabilidades a secuencias de símbolos y puede utilizarse para predecir o generar el siguiente símbolo. En este notebook los símbolos serán caracteres, aunque el mismo procedimiento puede aplicarse a palabras.

La regla de la cadena descompone la probabilidad de una secuencia $x_1, \ldots, x_T$ así:

$$ P(x_1, \ldots, x_T) = \prod_{t=1}^{T} P(x_t \mid x_1, \ldots, x_{t-1}) $$

El problema es que esta expresión necesita todo el contexto anterior. Los modelos de n-gramas realizan una aproximación de memoria limitada.

## **Modelos de n-gramas**

Un modelo de n-gramas estima el siguiente símbolo utilizando los $n-1$ símbolos anteriores:

$$ P(x_t \mid x_1, \ldots, x_{t-1}) \approx P(x_t \mid x_{t-n+1}, \ldots, x_{t-1}) $$

Por tanto:

$$ P(x_1, \ldots, x_T) \approx \prod_{t=1}^{T} P(x_t \mid x_{t-n+1}, \ldots, x_{t-1}) $$

La relación con los modelos de Markov es directa: un modelo de n-gramas es un modelo de Markov aplicado a una secuencia lingüística, con orden $n-1$.

| Modelo | Contexto utilizado | Orden de Markov |
|--------|--------------------|-----------------|
| Unigrama ($n=1$) | ningún símbolo anterior | 0 |
| Bigrama ($n=2$) | un símbolo anterior | 1 |
| Trigrama ($n=3$) | dos símbolos anteriores | 2 |

### **Tipos de modelos**

Un modelo de unigramas supone independencia entre símbolos:

$$ P(x_1, \ldots, x_T) \approx \prod_{t=1}^{T} P(x_t) $$

Un modelo de bigramas utiliza el símbolo anterior:

$$ P(x_1, \ldots, x_T) \approx \prod_{t=1}^{T} P(x_t \mid x_{t-1}) $$

Un modelo de trigramas utiliza los dos símbolos anteriores:

$$ P(x_1, \ldots, x_T) \approx \prod_{t=1}^{T} P(x_t \mid x_{t-2}, x_{t-1}) $$

### **Estimación de las probabilidades**

Las probabilidades se estiman contando los n-gramas del corpus. Para un modelo de n-gramas, la estimación de máxima verosimilitud es:

$$ P(x_t \mid x_{t-n+1}, \ldots, x_{t-1}) = \frac{C(x_{t-n+1}, \ldots, x_t)}{C(x_{t-n+1}, \ldots, x_{t-1})} $$

donde $C(\cdot)$ representa el número de apariciones en el corpus. Si un n-grama no aparece, esta estimación asigna probabilidad cero. En la práctica puede utilizarse suavizado para asignar una pequeña probabilidad a los casos no observados.

## **Entrenamiento y generación**

El entrenamiento consiste en contar símbolos y contextos en un corpus y normalizar esos conteos para obtener probabilidades. Una vez entrenado, el modelo genera una secuencia seleccionando cada nuevo carácter según la distribución asociada a su contexto.

En los modelos de n-gramas, aumentar $n$ permite utilizar más contexto, pero también aumenta el número de combinaciones posibles y la cantidad de n-gramas no observados. Esta es la razón por la que estos modelos tienen una memoria limitada y pueden necesitar suavizado.

### **Ventajas y limitaciones**

- **Simplicidad**: son fáciles de interpretar e implementar.
- **Eficiencia**: su entrenamiento y predicción suelen ser baratos.
- **Contexto limitado**: no capturan dependencias lejanas.
- **Escasez de datos**: al aumentar $n$, aparecen más contextos no observados.
- **Espacio de estados**: el número de contextos puede crecer rápidamente.

---

### **Ejercicio 1**

Construye modelos de lenguaje basados en unigramas, bigramas y trigramas para la novela `Cien años de soledad`, de Gabriel García Márquez. Tokeniza en función de los caracteres, no de las palabras. Después, genera texto con cada modelo y compara su perplejidad sobre una misma secuencia.

---

In [ ]:
import numpy as np
with open('novela.txt', 'r', encoding='utf-8') as fichero:
    text = fichero.read()

chars = sorted(set(text))
vocab_size = len(chars)
stoi = {char: index for index, char in enumerate(chars)}
itos = {index: char for index, char in enumerate(chars)}

print(f'Caracteres del corpus: {len(text):,}')
print(f'Tamaño del vocabulario: {vocab_size}')

### **Construcción de los tres modelos**

Se utiliza suavizado de Laplace con $\alpha=1$ para evitar probabilidades nulas. Así, todos los símbolos tienen una probabilidad pequeña incluso cuando un contexto concreto no aparece en el corpus.

In [ ]:
alpha = 1.0

# Modelo de unigramas: frecuencia de cada carácter.
unigram_counts = np.ones(vocab_size) * alpha
for char in text:
    unigram_counts[stoi[char]] += 1
unigram_probs = unigram_counts / unigram_counts.sum()

# Modelo de bigramas: carácter anterior -> carácter siguiente.
bigram_counts = np.ones((vocab_size, vocab_size)) * alpha
for current_char, next_char in zip(text, text[1:]):
    bigram_counts[stoi[current_char], stoi[next_char]] += 1
bigram_probs = bigram_counts / bigram_counts.sum(axis=1, keepdims=True)

# Modelo de trigramas: dos caracteres anteriores -> carácter siguiente.
trigram_counts = np.ones((vocab_size, vocab_size, vocab_size)) * alpha
for first_char, second_char, next_char in zip(text, text[1:], text[2:]):
    trigram_counts[stoi[first_char], stoi[second_char], stoi[next_char]] += 1
trigram_probs = trigram_counts / trigram_counts.sum(axis=2, keepdims=True)

print('Modelos entrenados.')

### **Generación de texto**

La función siguiente selecciona caracteres aleatoriamente según las probabilidades del modelo correspondiente. Para el trigrama, el contexto está formado por los dos últimos caracteres generados.

In [ ]:
def generate_text(model, number_of_chars, initial_text='C'):
    generated = list(initial_text)

    for _ in range(number_of_chars):
        if model == 'unigram':
            probabilities = unigram_probs
        elif model == 'bigram':
            probabilities = bigram_probs[stoi[generated[-1]]]
        elif model == 'trigram':
            if len(generated) < 2:
                probabilities = bigram_probs[stoi[generated[-1]]]
            else:
                probabilities = trigram_probs[stoi[generated[-2]], stoi[generated[-1]]]
        else:
            raise ValueError('El modelo debe ser unigram, bigram o trigram')

        next_index = np.random.choice(vocab_size, p=probabilities)
        generated.append(itos[next_index])

    return ''.join(generated)

for model_name in ('unigram', 'bigram', 'trigram'):
    print(f'\n{model_name}:')
    print(generate_text(model_name, 200, 'C'))

## **Perplejidad**

La perplejidad mide la capacidad del modelo para asignar probabilidad a una secuencia observada. En este notebook, $x_t$ representa un carácter y no una palabra. Para una secuencia de $T$ caracteres:

$$ \operatorname{PP}(x_1, \ldots, x_T) = \exp\left(-\frac{1}{T} \sum_{t=1}^{T} \log P(x_t \mid contexto_t)\right) $$

Se utiliza el logaritmo natural en la implementación. Una perplejidad menor indica que el modelo asigna, en promedio, mayor probabilidad a los caracteres observados. La comparación solo es válida si todos los modelos se evalúan sobre la misma secuencia y con la misma convención de cálculo.

In [ ]:
def perplexity(model, sequence):
    log_probability = 0.0
    number_of_predictions = 0

    for index, current_char in enumerate(sequence):
        if current_char not in stoi:
            raise ValueError(f'El carácter {current_char!r} no aparece en el corpus')

        current_index = stoi[current_char]
        if model == 'unigram':
            probability = unigram_probs[current_index]
        elif model == 'bigram':
            if index == 0:
                probability = unigram_probs[current_index]
            else:
                previous_index = stoi[sequence[index - 1]]
                probability = bigram_probs[previous_index, current_index]
        elif model == 'trigram':
            if index == 0:
                probability = unigram_probs[current_index]
            elif index == 1:
                previous_index = stoi[sequence[index - 1]]
                probability = bigram_probs[previous_index, current_index]
            else:
                first_index = stoi[sequence[index - 2]]
                second_index = stoi[sequence[index - 1]]
                probability = trigram_probs[first_index, second_index, current_index]
        else:
            raise ValueError('El modelo debe ser unigram, bigram o trigram')

        log_probability += np.log(probability)
        number_of_predictions += 1

    return np.exp(-log_probability / number_of_predictions)

In [ ]:
sequence = 'Muchos años después, frente al pelotón de fusilamiento, el coronel Aureliano Buendía'

for model_name in ('unigram', 'bigram', 'trigram'):
    value = perplexity(model_name, sequence)
    print(f'{model_name:>7}: {value:.3f}')

print('\nSecuencia invertida:')
reversed_sequence = sequence[::-1]
for model_name in ('unigram', 'bigram', 'trigram'):
    value = perplexity(model_name, reversed_sequence)
    print(f'{model_name:>7}: {value:.3f}')

### **Ejercicio 2**

Interpreta los resultados obtenidos. ¿Qué modelo obtiene menor perplejidad? ¿Cómo cambia la perplejidad al invertir la secuencia? Relaciona la respuesta con la cantidad de contexto que utiliza cada modelo y con la dirección en la que se aprendieron las transiciones.